In [1]:
import pandas as pd


df = pd.read_csv("C:/Users/MY PC/OneDrive/Desktop/retrain/Reviews.csv")

In [25]:
df = df.drop_duplicates()

In [26]:
df.shape


(568454, 10)

In [27]:
print(df.columns.tolist())

['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


In [28]:
df=df.drop(columns=['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator','Time', 'Summary'])

In [29]:
print(df.columns.tolist())

['Score', 'Text']


In [30]:
df = df.rename(columns={"Text": "Review", "Score": "Rating"})
print(df.head())

   Rating                                             Review
0       5  I have bought several of the Vitality canned d...
1       1  Product arrived labeled as Jumbo Salted Peanut...
2       4  This is a confection that has been around a fe...
3       2  If you are looking for the secret ingredient i...
4       5  Great taffy at a great price.  There was a wid...


In [31]:
df = df.drop_duplicates()

In [32]:
rating_counts = df['Rating'].value_counts().sort_index()
rating_counts

Rating
1     36280
2     20804
3     29772
4     56074
5    250745
Name: count, dtype: int64

In [33]:
print("🔍 Checking for missing values...")
print(df.isnull().sum())

🔍 Checking for missing values...
Rating    0
Review    0
dtype: int64


In [34]:
# Display only the rows with missing values
df[df.isnull().any(axis=1)]


,Rating,Review


In [35]:
df.head

<bound method NDFrame.head of         Rating                                             Review
0            5  I have bought several of the Vitality canned d...
1            1  Product arrived labeled as Jumbo Salted Peanut...
2            4  This is a confection that has been around a fe...
3            2  If you are looking for the secret ingredient i...
4            5  Great taffy at a great price.  There was a wid...
...        ...                                                ...
568449       5  Great for sesame chicken..this is a good if no...
568450       2  I'm disappointed with the flavor. The chocolat...
568451       5  These stars are small, so you can give 10-15 o...
568452       5  These are the BEST treats for training and rew...
568453       5  I am very satisfied ,product is as advertised,...

[393675 rows x 2 columns]>

In [37]:
df.head()


,Rating,Review
0,5,I have bought several of the Vitality canned d...
1,1,Product arrived labeled as Jumbo Salted Peanut...
2,4,This is a confection that has been around a fe...
3,2,If you are looking for the secret ingredient i...
4,5,Great taffy at a great price. There was a wid...


In [38]:
# Check if any reviews are repeated
duplicate_reviews = df[df.duplicated(subset=['Review'], keep=False)]

print("🔍 Total duplicate reviews:", df.duplicated(subset=['Review']).sum())
duplicate_reviews.head()


🔍 Total duplicate reviews: 96


,Rating,Review
1791,1,I don't know why everyone loves these chips. T...
2806,2,"When I first started to use IAMS Savory Sauce,..."
4928,3,Buffalo Bills Premium Snacks<br /><br />I real...
5023,4,Good product! good price fast shipping . need...
6000,5,"A few years ago, my cat was diagnosed with foo..."


In [39]:
df = df.drop_duplicates(subset=['Review', 'Rating'])
print("✅ Duplicates removed. New shape:", df.shape)


✅ Duplicates removed. New shape: (393675, 2)


In [40]:
# Check if any reviews are repeated
duplicate_reviews = df[df.duplicated(subset=['Review'], keep=False)]

print("🔍 Total duplicate reviews:", df.duplicated(subset=['Review']).sum())
duplicate_reviews.head()


🔍 Total duplicate reviews: 96


,Rating,Review
1791,1,I don't know why everyone loves these chips. T...
2806,2,"When I first started to use IAMS Savory Sauce,..."
4928,3,Buffalo Bills Premium Snacks<br /><br />I real...
5023,4,Good product! good price fast shipping . need...
6000,5,"A few years ago, my cat was diagnosed with foo..."


In [41]:
df = df.drop_duplicates(subset=['Review', 'Rating'])
print("✅ Duplicates removed. New shape:", df.shape)


✅ Duplicates removed. New shape: (393675, 2)


In [43]:
# Remove duplicate reviews (keep only the first occurrence)
df= df.drop_duplicates(subset=['Review'], keep='first')

print("✅ All duplicate reviews removed.")
print("Remaining rows:", df.shape[0])


✅ All duplicate reviews removed.
Remaining rows: 393579


In [44]:
# Double-check that no duplicate reviews remain
print("🔍 Remaining duplicate reviews:", df.duplicated(subset=['Review']).sum())


🔍 Remaining duplicate reviews: 0


In [45]:
df.shape()

TypeError: 'tuple' object is not callable

In [46]:
df.shape


(393579, 2)

In [47]:
df.head()

,Rating,Review
0,5,I have bought several of the Vitality canned d...
1,1,Product arrived labeled as Jumbo Salted Peanut...
2,4,This is a confection that has been around a fe...
3,2,If you are looking for the secret ingredient i...
4,5,Great taffy at a great price. There was a wid...


In [48]:
rating_counts = df['Rating'].value_counts().sort_index()
rating_counts

Rating
1     36275
2     20792
3     29754
4     56042
5    250716
Name: count, dtype: int64

In [64]:
import pandas as pd

df = df.copy()  # original dataset

# Step 1: Sample imbalanced dataset
target_counts = {
    1: 2500,
    2: 3750,
    3: 6250,
    4: 7500,
    5: 5000
}

sampled_dfs = []
for rating, count in target_counts.items():
    subset = df[df['Rating'] == rating]
    sampled = subset.sample(n=count, random_state=42,
                            replace=True if len(subset) < count else False)
    sampled_dfs.append(sampled)

imbalanced_df = pd.concat(sampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

# Step 2: Create remaining dataset correctly
# Use merge with indicator to avoid index issues
remaining_df = df.merge(imbalanced_df, on=['Rating','Review'], how='outer', indicator=True)
remaining_df = remaining_df[remaining_df['_merge'] == 'left_only'].drop(columns=['_merge'])

# Save both datasets
imbalanced_path = r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_imbalanced.csv"
imbalanced_df.to_csv(imbalanced_path, index=False)

remaining_path = r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_remaining.csv"
remaining_df.to_csv(remaining_path, index=False)

print(f"✅ Imbalanced dataset shape: {imbalanced_df.shape}")
print(f"✅ Remaining dataset shape: {remaining_df.shape}")


✅ Imbalanced dataset shape: (25000, 2)
✅ Remaining dataset shape: (368579, 2)


In [66]:
df1=pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_imbalanced.csv")
df2=pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_remaining.csv")


In [67]:
import pandas as pd

# Load datasets
df1 = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_imbalanced.csv")
df2 = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_remaining.csv")

# Check overlap
overlap_reviews = set(df1['Review']).intersection(set(df2['Review']))

if len(overlap_reviews) == 0:
    print("✅ No overlapping reviews. Datasets are mutually exclusive.")
else:
    print(f"⚠️ Found {len(overlap_reviews)} overlapping reviews between the two datasets.")


✅ No overlapping reviews. Datasets are mutually exclusive.


In [68]:
df=pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_imbalanced.csv")

In [69]:
import pandas as pd

# Load the dataset
df = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_imbalanced.csv")

# Find rows where 'Review' is NaN
nan_reviews = df[df['Review'].isna()]

# Show count and examples
print(f"Total reviews with NaN text: {nan_reviews.shape[0]}")
print(nan_reviews.head())


Total reviews with NaN text: 0
Empty DataFrame
Columns: [Rating, Review]
Index: []


In [70]:
import pandas as pd

# Load dataset
df = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_imbalanced.csv")

# Filter reviews with Rating 3
rating_3_reviews = df[df['Rating'] == 3]

# Print all review texts and their ratings
for index, row in rating_3_reviews.iterrows():
    print(f"Rating: {row['Rating']}, Review: {row['Review']}")


Rating: 3, Review: I used this for a month, but not sure that it really helps me. When my baby started to burp up milk, I stopped using it to eliminate the cause. I have not gone back to using the fenugreek because I now have more milk than when I was taking the fenugreek. Hence, I am not sure whether this helps or not, but I would not discourage people from using it since many of my friends have used it with successful.
Rating: 3, Review: So I've been looking for this cereal literally every time I go to the store ever since I had it at boot camp 5 months ago. I went to about 4 to 5 different stores all with no success, also none of the store associates from any of the stores knew what I was talking about. Least to say, I was extremely happy to see that I could order my dream cereal on here. The product description does say that its 50 oz. however, for some reason I thought I would receive a much bigger bag for the price of $17.47. The cereal, like I remember it, is very good and my bo

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [71]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

In [72]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = text.lower().split()                 # lowercase + tokenize
    tokens = [w for w in tokens if w.isalpha()]  # keep only words
    tokens = [w for w in tokens if w not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(w) for w in tokens]   # lemmatization
    return " ".join(tokens)


In [73]:
df['Cleaned_Text'] = df['Review'].astype(str).apply(preprocess)

# Check the cleaned text
df[['Review','Cleaned_Text']].head()


,Review,Cleaned_Text
0,"I used this for a month, but not sure that it ...",used sure really help baby started burp stoppe...
1,You can't tell this from the wheat brands. Mix...,tell wheat mix make wonderful moist sometimes ...
2,So I've been looking for this cereal literally...,looking cereal literally every time go store e...
3,This sauce is excellent. Medium hot with smoky...,sauce medium hot smoky undertone distinctive g...
4,"Although this cocoa is called Special Dark, it...",although cocoa called special special dark coc...


In [74]:

df.columns


Index(['Rating', 'Review', 'Cleaned_Text'], dtype='object')

In [75]:

df = df.drop(columns=['Review'])

# Check remaining columns
df.columns


Index(['Rating', 'Cleaned_Text'], dtype='object')

In [76]:

df = df.rename(columns={'Cleaned_Text': 'Review'})


df.columns


Index(['Rating', 'Review'], dtype='object')

In [77]:
df.columns

Index(['Rating', 'Review'], dtype='object')

In [78]:
# Remove any leading/trailing spaces in column names
df.columns = df.columns.str.strip()

# Check columns again
print(df.columns)


Index(['Rating', 'Review'], dtype='object')


In [80]:
# Print first 20 reviews with Rating 3
print(rating_3_reviews.head(6250))


       Rating                                             Review
0           3  I used this for a month, but not sure that it ...
2           3  So I've been looking for this cereal literally...
5           3  The product is good but one three of the bags ...
9           3  I found the flavor to be very unusual: almost ...
12          3  The texture is great and I suppose if you like...
...       ...                                                ...
24984       3  I was expecting a really spectacular cup of te...
24986       3  The coffee is very good, but it seems the pric...
24991       3  I was delighted to discover this product whils...
24993       3  I've tried quite a few of this Switch juice dr...
24994       3  I grew up with Kool Aid, during an era where y...

[6250 rows x 2 columns]


In [81]:
print(df.isna().sum())

Rating    0
Review    0
dtype: int64


In [82]:
from sklearn.model_selection import train_test_split

# Ensure column names have no extra spaces
df.columns = df.columns.str.strip()

# Features and target
X = df['Review']
y = df['Rating']

# Stratified train-test split (80%-20%)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Verify class distribution in train and test sets
train_counts = y_train.value_counts().sort_index()
test_counts  = y_test.value_counts().sort_index()

train_percent = y_train.value_counts(normalize=True).sort_index() * 100
test_percent  = y_test.value_counts(normalize=True).sort_index() * 100

print("=== Training Set Distribution ===")
print(pd.DataFrame({'Count': train_counts, 'Percentage': train_percent}))

print("\n=== Testing Set Distribution ===")
print(pd.DataFrame({'Count': test_counts, 'Percentage': test_percent}))



=== Training Set Distribution ===
        Count  Percentage
Rating                   
1        2000        10.0
2        3000        15.0
3        5000        25.0
4        6000        30.0
5        4000        20.0

=== Testing Set Distribution ===
        Count  Percentage
Rating                   
1         500        10.0
2         750        15.0
3        1250        25.0
4        1500        30.0
5        1000        20.0


In [83]:
# Check NaN values in X_train_text and X_test_text
print("NaN values in X_train_text:", X_train_text.isna().sum())
print("NaN values in X_test_text :", X_test_text.isna().sum())

# Check NaN values in y_train and y_test
print("NaN values in y_train:", y_train.isna().sum())
print("NaN values in y_test :", y_test.isna().sum())


NaN values in X_train_text: 0
NaN values in X_test_text : 0
NaN values in y_train: 0
NaN values in y_test : 0


In [84]:
# Combine X_test_text and y_test into a single DataFrame
test_df = pd.DataFrame({
    'Review': X_test_text,
    'Rating': y_test
})

# Filter rows where Rating == 3
rating_3_test = test_df[test_df['Rating'] == 3]

# Print all reviews with Rating 3
for i, row in enumerate(rating_3_test.itertuples(), 1):
    print(f"{i}. Rating: {row.Rating}, Review: {row.Review}")


1. Rating: 3, Review: great price becoming harder harder find local like full many graham cracker kid love disappointment eighth package crumbled little ship
2. Rating: 3, Review: description balsamic indeed valuable review fails note unlike fine also grows ever one nip unfit drive put know going may able withstand effect heady take note two year shelf two enough oxygen undoubtedly penetrated turn vinegar good fine aged blend vein two buck use sparingly use buy slightly cheaper burning
3. Rating: 3, Review: product sweet qualify real baby nutrition many carbs ingredient pouch bpa free also good pouch easy squeeze could choking hazard baby pouch would good qualifier quick carb boost casual jogger used pouch would easy flatten pack back home properly dispose cap nice sweet good fun meal thriled role nutritious baby
4. Rating: 3, Review: using brand dog food two appears good dog problem additional amount paid delivered saturday monday still receive refund expedited delivery return amend
5

In [85]:
import pandas as pd

# Combine X_train_text and y_train into a single DataFrame
train_df = pd.DataFrame({
    'Review': X_train_text,
    'Rating': y_train
})

# Combine X_test_text and y_test into a single DataFrame
test_df = pd.DataFrame({
    'Review': X_test_text,
    'Rating': y_test
})

# Save the datasets
train_path = r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_train_data.csv"
test_path  = r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_test_data.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"✅ Training dataset saved at {train_path} with shape {train_df.shape}")
print(f"✅ Testing dataset saved at {test_path} with shape {test_df.shape}")


✅ Training dataset saved at C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_train_data.csv with shape (20000, 2)
✅ Testing dataset saved at C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_test_data.csv with shape (5000, 2)


In [86]:
df= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_train_data.csv")
df= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_test_data.csv")

In [2]:
import pandas as pd

# Load datasets
df= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_train_data.csv")
df= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_test_data.csv")

# Check for NaN values in train dataset
print("=== NaN values in Train Dataset ===")
print(train_df.isna().sum())
print(f"Total NaN values in Train Dataset: {train_df.isna().sum().sum()}\n")

# Check for NaN values in test dataset
print("=== NaN values in Test Dataset ===")
print(test_df.isna().sum())
print(f"Total NaN values in Test Dataset: {test_df.isna().sum().sum()}")


=== NaN values in Train Dataset ===


NameError: name 'train_df' is not defined

In [3]:
import pandas as pd

# Load datasets
train_df = = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/orginal/Imb_train_data.csv")
test_df  = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_test_data.csv")

# Check for NaN values in train dataset
print("=== NaN values in Train Dataset ===")
print(train_df.isna().sum())
print(f"Total NaN values in Train Dataset: {train_df.isna().sum().sum()}\n")

# Check for NaN values in test dataset
print("=== NaN values in Test Dataset ===")
print(test_df.isna().sum())
print(f"Total NaN values in Test Dataset: {test_df.isna().sum().sum()}")


=== NaN values in Train Dataset ===
Review    1
Rating    0
dtype: int64
Total NaN values in Train Dataset: 1

=== NaN values in Test Dataset ===
Review    0
Rating    0
dtype: int64
Total NaN values in Test Dataset: 0


In [4]:
import pandas as pd

# Load training dataset
train_df = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Imb_train_data.csv")

# Replacement text for missing 4-star review
replacement_review = "good service and satisfactory experience"

# Replace the missing review where Rating == 4
train_df.loc[train_df['Review'].isna() & (train_df['Rating'] == 4), 'Review'] = replacement_review

# Verify replacement
missing_after = train_df[train_df['Review'].isna() & (train_df['Rating'] == 4)]
print(f"Number of missing 4-star reviews after replacement: {len(missing_after)}")

# Save the cleaned dataset in the same folder as original
cleaned_file_path = r"C:/Users/MY PC/OneDrive/Desktop/orginal/Imb_train_data.csv"
train_df.to_csv(cleaned_file_path, index=False)
print(f"✅ Cleaned training dataset saved at: {cleaned_file_path}")


Number of missing 4-star reviews after replacement: 0
✅ Cleaned training dataset saved at: C:/Users/MY PC/OneDrive/Desktop/orginal/Imb_train_data.csv


In [ ]:
= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/orginal/Imb_train_data.csv")